# Proyecto 2 - Modelado Predictivo de Mortalidad (Fases 0-5)

Este notebook implementa las primeras 5 fases del plan de prediccion y mantiene separado el Proyecto 1 (EDA/Clustering) del Proyecto 2 (Modelos).

Fases cubiertas aqui:
- Fase 0: Reutilizacion controlada
- Fase 1: Definir variable objetivo
- Fase 2: Antecedentes (plantilla academica)
- Fase 3: Preparacion de datos
- Fase 4: Train/Validation/Test split
- Fase 5: Seleccion de algoritmos


## Fase 0 - Reutilizacion controlada

Este notebook reutiliza logica de limpieza ya alineada con el Proyecto 1 (ejemplo: tratamiento de `Edadif == 999` y manejo de columnas entre anos).

Se mantiene separado de `main.ipynb` para:
- evitar notebooks demasiado largos,
- no afectar el flujo EDA/Clustering,
- facilitar revision independiente en GitHub y por el profesor.


In [42]:
from __future__ import annotations

from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import pyreadstat

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [43]:
RANDOM_STATE = 42
YEARS = list(range(2013, 2023))
DATA_DIR = Path('data/defunciones')

# Opcional para prototipar rapido en equipos con menos RAM
USE_SAMPLE = False
SAMPLE_SIZE = 200_000

np.random.seed(RANDOM_STATE)

In [44]:
def normalize_text(value: str) -> str:
    if value is None:
        return ''
    value = unicodedata.normalize('NFKD', str(value))
    value = ''.join(ch for ch in value if not unicodedata.combining(ch))
    return value.lower().strip()


def resolve_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    norm_map = {normalize_text(col): col for col in df.columns}
    for candidate in candidates:
        key = normalize_text(candidate)
        if key in norm_map:
            return norm_map[key]
    return None


def normalize_cie10(code: object) -> str | None:
    if pd.isna(code):
        return None
    text = str(code).upper().strip()
    # Captura codigos tipo A15, I21, Y87.0
    match = re.search(r'[A-Z][0-9]{2}(?:\.[0-9])?', text)
    return match.group(0) if match else None


def _cie_num(cie10: str) -> int | None:
    try:
        return int(cie10[1:3])
    except Exception:
        return None


def map_causa_grupo(cie10: str | None) -> str | None:
    """
    Mapeo detallado inspirado en diccionario.md, con clases no traslapadas
    para clasificacion supervisada.
    """
    if cie10 is None:
        return None

    letter = cie10[0]
    num = _cie_num(cie10)

    # Infecciosas
    if letter in {'A', 'B'}:
        return 'Infecciosas'

    # Neoplasias / sangre e inmunidad
    if letter == 'C':
        return 'Neoplasias'
    if letter == 'D' and num is not None:
        if 0 <= num <= 48:
            return 'Neoplasias'
        if 50 <= num <= 89:
            return 'Sangre_inmunidad'

    # Endocrinas, mentales, nervioso/sentidos
    if letter == 'E':
        return 'Endocrinas_metabolicas'
    if letter == 'F':
        return 'Trastornos_mentales'
    if letter in {'G', 'H'}:
        return 'Nervioso_organos_sentidos'

    # Circulatorias (subgrupos del diccionario)
    if letter == 'I' and num is not None:
        if 10 <= num <= 13:
            return 'Hipertensiva'
        if 20 <= num <= 25:
            return 'Isquemica_corazon'
        if 60 <= num <= 69:
            return 'Cerebrovascular'
        return 'Otras_circulatorias'

    # Respiratorias (subgrupos del diccionario)
    if letter == 'J' and num is not None:
        if 10 <= num <= 18:
            return 'Neumonia_influenza'
        if 40 <= num <= 47:
            return 'EPOC'
        return 'Otras_respiratorias'

    # Digestivas (subgrupos del diccionario)
    if letter == 'K' and num is not None:
        if num == 70 or 73 <= num <= 74:
            return 'Cronica_higado'
        if 35 <= num <= 38:
            return 'Apendicitis'
        if (40 <= num <= 46) or num == 56:
            return 'Hernia_obstruccion_intestinal'
        return 'Otras_digestivas'

    # Genitourinarias (subgrupos del diccionario)
    if letter == 'N' and num is not None:
        if (0 <= num <= 7) or (17 <= num <= 19) or (25 <= num <= 27):
            return 'Nefritis_sindrome_nefrotico'
        return 'Otras_genitourinarias'

    # Causas externas (subgrupos del diccionario)
    if letter == 'V' and num is not None:
        if 2 <= num <= 89:
            return 'Accidentes_transito'
        return 'Accidentes_no_intencionales'

    if letter == 'W':
        return 'Accidentes_no_intencionales'

    if letter == 'X' and num is not None:
        if 60 <= num <= 84:
            return 'Suicidio'
        if 85 <= num <= 99:
            return 'Homicidio'
        return 'Accidentes_no_intencionales'

    if letter == 'Y' and num is not None:
        if 0 <= num <= 9:
            return 'Homicidio'
        if 40 <= num <= 59:
            return 'Efectos_adversos_medicamentos'
        if 85 <= num <= 86:
            return 'Accidentes_no_intencionales'
        return 'Otras_causas_externas'

    # Separaciones clave para no perder informacion
    if letter == 'R':
        return 'Sintomas_signos_hallazgos'
    if letter == 'Q':
        return 'Congenitas'
    if letter == 'P':
        return 'Perinatales'
    if letter == 'O':
        return 'Embarazo_parto_puerperio'
    if letter == 'U':
        return 'Codigos_especiales'

    # Otros capitulos CIE-10
    if letter in {'L', 'M', 'Z'}:
        return 'Otros_capitulos'

    return 'Otros_capitulos'


def load_yearly_data(data_dir: Path, years: list[int]) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for year in years:
        path = data_dir / f'{year}.sav'
        if not path.exists():
            print(f'[WARN] No existe: {path}')
            continue
        df_year, _ = pyreadstat.read_sav(str(path))
        df_year['__year_file__'] = year
        frames.append(df_year)
        print(f'[OK] {year}: {len(df_year):,} registros')

    if not frames:
        raise FileNotFoundError('No se pudieron cargar archivos .sav en data/defunciones')

    return pd.concat(frames, ignore_index=True)


In [45]:
df = load_yearly_data(DATA_DIR, YEARS)
print(f'\nShape consolidado: {df.shape}')

if USE_SAMPLE and len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f'Se usa muestra para prototipado: {df.shape}')

[OK] 2013: 76,639 registros
[OK] 2014: 77,807 registros
[OK] 2015: 80,876 registros
[OK] 2016: 82,565 registros
[OK] 2017: 81,726 registros
[OK] 2018: 83,071 registros
[OK] 2019: 85,600 registros
[OK] 2020: 96,001 registros
[OK] 2021: 118,465 registros
[OK] 2022: 95,386 registros

Shape consolidado: (878136, 30)


## Fase 1 - Variable objetivo

**Variable respuesta seleccionada:** `grupo_causa_cie10` (derivada de `Caudef` CIE-10).

- Tipo: **cualitativa nominal** (clasificacion multiclase).
- Enfoque: mapeo **detallado** inspirado en `diccionario.md`, no consolidado.
- Clases principales consideradas:
  - `Infecciosas`, `Neoplasias`, `Sangre_inmunidad`, `Endocrinas_metabolicas`, `Trastornos_mentales`, `Nervioso_organos_sentidos`
  - `Hipertensiva`, `Isquemica_corazon`, `Cerebrovascular`, `Otras_circulatorias`
  - `Neumonia_influenza`, `EPOC`, `Otras_respiratorias`
  - `Cronica_higado`, `Apendicitis`, `Hernia_obstruccion_intestinal`, `Otras_digestivas`
  - `Nefritis_sindrome_nefrotico`, `Otras_genitourinarias`
  - `Accidentes_transito`, `Accidentes_no_intencionales`, `Suicidio`, `Homicidio`, `Efectos_adversos_medicamentos`, `Otras_causas_externas`
  - `Sintomas_signos_hallazgos`, `Congenitas`, `Perinatales`, `Embarazo_parto_puerperio`, `Codigos_especiales`, `Otros_capitulos`

Fuente: `diccionario.md` (tabla CIE-10) y estructura de capitulos ICD-10 OMS.

Nota metodologica: la clase `Sintomas_signos_hallazgos` puede sugerir problemas de calidad de certificacion o diagnostico, pero no prueba por si sola negligencia medica.


In [46]:
caudef_col = resolve_column(df, ['Caudef'])
if caudef_col is None:
    raise KeyError('No se encontro la columna Caudef para construir la variable objetivo.')

df['cie10_norm'] = df[caudef_col].apply(normalize_cie10)
df['grupo_causa_cie10'] = df['cie10_norm'].apply(map_causa_grupo)

print('Distribucion inicial de la variable objetivo:')
display(df['grupo_causa_cie10'].value_counts(dropna=False).to_frame('conteo'))

df = df[df['grupo_causa_cie10'].notna()].copy()
print(f'\nRegistros tras remover objetivo nulo: {len(df):,}')

Distribucion inicial de la variable objetivo:


,conteo
grupo_causa_cie10,
Sintomas_signos_hallazgos,103142
Endocrinas_metabolicas,91789
Neoplasias,84843
Isquemica_corazon,68284
Neumonia_influenza,62336
Accidentes_no_intencionales,60300
Infecciosas,41420
Homicidio,34397
Cronica_higado,34237



Registros tras remover objetivo nulo: 878,136


## Diagnostico de `Sintomas_signos_hallazgos` (R) y `Otros_capitulos`

Se analiza por separado la clase R para no perder señal relevante del contexto de calidad diagnostica.


In [47]:
def resumen_clase(clase: str) -> None:
    mask = df['grupo_causa_cie10'].eq(clase)
    df_sub = df.loc[mask, ['cie10_norm', caudef_col, '__year_file__']].copy()

    print(f'=== {clase} ===')
    if df_sub.empty:
        print('Sin registros.')
        return

    total = len(df)
    n = len(df_sub)
    print(f'Registros: {n:,} ({n / total:.2%} del total)')

    df_sub['cie10_letra'] = df_sub['cie10_norm'].str[0]

    print()
    print('Top 15 codigos CIE-10:')
    display(df_sub['cie10_norm'].value_counts().head(15).to_frame('conteo'))

    print()
    print('Distribucion por letra CIE-10:')
    display(df_sub['cie10_letra'].value_counts().to_frame('conteo'))

    print()
    print('Distribucion por anio y letra (% fila):')
    tabla = pd.crosstab(df_sub['__year_file__'], df_sub['cie10_letra'], normalize='index').mul(100).round(2)
    display(tabla)

resumen_clase('Sintomas_signos_hallazgos')
print()
resumen_clase('Otros_capitulos')


=== Sintomas_signos_hallazgos ===
Registros: 103,142 (11.75% del total)

Top 15 codigos CIE-10:


,conteo
cie10_norm,
R98,34582
R99,27258
R54,23587
R68,2688
R50,2589
R57,2393
R09,2358
R95,1684
R56,1666



Distribucion por letra CIE-10:


,conteo
cie10_letra,
R,103142



Distribucion por anio y letra (% fila):


...

,conteo
cie10_norm,
M06,763
L98,618
M32,548
L89,354
M13,327
M19,323
M81,282
L08,243
M79,185



Distribucion por letra CIE-10:


,conteo
cie10_letra,
M,3352
L,1658



Distribucion por anio y letra (% fila):


cie10_letra,L,M
__year_file__,,
2013,17.42,82.58
2014,16.87,83.13
2015,37.73,62.27
2016,40.42,59.58
2017,33.40,66.60
2018,40.57,59.43
2019,37.43,62.57
2020,28.17,71.83
2021,35.22,64.78


## Interpretacion de la clase R (sintomas/signos sin diagnostico definitivo)

Esta seccion resume que significa la clase `Sintomas_signos_hallazgos` (capitulo R de CIE-10) y cuales son sus codigos mas frecuentes en el dataset.

Lectura epidemiologica sugerida:
- Los codigos R suelen indicar que no se establecio una causa clinica final especifica.
- Un volumen alto puede reflejar limites de certificacion, registro o disponibilidad diagnostica.
- No implica por si solo negligencia medica, pero es una senal importante de calidad de datos a vigilar.


In [48]:
# Descripciones breves para codigos R mas frecuentes (referencia CIE-10)
R_DESCRIPTIONS = {
    'R98': 'Muerte sin asistencia (unattended death)',
    'R99': 'Otras causas mal definidas o no especificadas de mortalidad',
    'R54': 'Senilidad',
    'R68': 'Otros sintomas y signos generales',
    'R50': 'Fiebre de origen desconocido',
    'R57': 'Choque (shock), no clasificado en otra parte',
    'R09': 'Otros sintomas del sistema circulatorio y respiratorio',
}

df_r = df[df['grupo_causa_cie10'] == 'Sintomas_signos_hallazgos'].copy()

if df_r.empty:
    print('No hay registros clasificados como Sintomas_signos_hallazgos (R).')
else:
    top_r = df_r['cie10_norm'].value_counts().head(12).rename('conteo').reset_index()
    top_r = top_r.rename(columns={'index': 'codigo_r', 'cie10_norm': 'codigo_r'})
    if 'codigo_r' not in top_r.columns:
        # compatibilidad por versiones de pandas
        top_r.columns = ['codigo_r', 'conteo']

    top_r['descripcion'] = top_r['codigo_r'].map(R_DESCRIPTIONS).fillna('Descripcion CIE-10 no cargada en este resumen')
    top_r['porcentaje_dentro_R'] = (top_r['conteo'] / len(df_r) * 100).round(2)

    print(f'Registros totales en R: {len(df_r):,}')
    print('Top codigos R y su interpretacion:')
    display(top_r[['codigo_r', 'conteo', 'porcentaje_dentro_R', 'descripcion']])

    p_top3 = (top_r['conteo'].head(3).sum() / len(df_r) * 100)
    print(f'Los 3 codigos R mas frecuentes explican {p_top3:.2f}% de la clase R.')


Registros totales en R: 103,142
Top codigos R y su interpretacion:


,codigo_r,conteo,porcentaje_dentro_R,descripcion
0,R98,34582,33.53,Muerte sin asistencia (unattended death)
1,R99,27258,26.43,Otras causas mal definidas o no especificadas ...
2,R54,23587,22.87,Senilidad
3,R68,2688,2.61,Otros sintomas y signos generales
4,R50,2589,2.51,Fiebre de origen desconocido
5,R57,2393,2.32,"Choque (shock), no clasificado en otra parte"
6,R09,2358,2.29,Otros sintomas del sistema circulatorio y resp...
7,R95,1684,1.63,Descripcion CIE-10 no cargada en este resumen
8,R56,1666,1.62,Descripcion CIE-10 no cargada en este resumen
9,R10,1299,1.26,Descripcion CIE-10 no cargada en este resumen


Los 3 codigos R mas frecuentes explican 82.82% de la clase R.


## Fase 3 - Preparacion de datos

En esta fase se construye una matriz de modelado sin fuga de informacion:
- no se usa `Caudef` como feature (origen directo de la etiqueta),
- se limpian sentinelas (`Edadif == 999`),
- se generan variables temporales simples para mejorar senal predictiva.


In [49]:
sexo_col = resolve_column(df, ['Sexo'])
edad_col = resolve_column(df, ['Edadif'])
depocu_col = resolve_column(df, ['Depocu', 'Depreg'])
mupocu_col = resolve_column(df, ['Mupocu', 'Mupreg'])
mes_col = resolve_column(df, ['Mesocu', 'Mesreg'])
dia_col = resolve_column(df, ['Diaocu'])
anio_col = resolve_column(df, ['Añoocu', 'Anoocu', 'Añoreg', 'Anoreg'])
ecivil_col = resolve_column(df, ['Ecidif'])
escolar_col = resolve_column(df, ['Escodif'])
ocup_col = resolve_column(df, ['Ciuodif'])

feature_map = {
    'sexo': sexo_col,
    'edad': edad_col,
    'departamento': depocu_col,
    'municipio': mupocu_col,
    'mes': mes_col,
    'dia': dia_col,
    'anio': anio_col,
    'estado_civil': ecivil_col,
    'escolaridad': escolar_col,
    'ocupacion': ocup_col,
}

missing_features = [k for k, v in feature_map.items() if v is None]
print('Columnas detectadas:')
display(pd.Series(feature_map, name='columna_en_dataset'))
if missing_features:
    print(f'[WARN] No detectadas: {missing_features}')

selected_pairs = [(k, v) for k, v in feature_map.items() if v is not None]
df_model = df[[v for _, v in selected_pairs]].copy()
df_model.columns = [k for k, _ in selected_pairs]

# Limpieza principal heredada de practicas del Proyecto 1
if 'edad' in df_model.columns:
    df_model['edad'] = pd.to_numeric(df_model['edad'], errors='coerce')
    df_model.loc[df_model['edad'] == 999, 'edad'] = np.nan

if 'dia' in df_model.columns:
    df_model['dia'] = pd.to_numeric(df_model['dia'], errors='coerce')

if 'mes' in df_model.columns:
    df_model['mes'] = pd.to_numeric(df_model['mes'], errors='coerce')

if 'anio' in df_model.columns:
    df_model['anio'] = pd.to_numeric(df_model['anio'], errors='coerce')

# Feature temporal simple
if {'dia', 'mes', 'anio'}.issubset(df_model.columns):
    fecha = pd.to_datetime(
        dict(year=df_model['anio'], month=df_model['mes'], day=df_model['dia']),
        errors='coerce',
    )
    df_model['es_fin_semana'] = fecha.dt.dayofweek.isin([5, 6]).astype('float')

y = df['grupo_causa_cie10'].copy()

print(f'Shape de modelado: X={df_model.shape}, y={y.shape}')

Columnas detectadas:


sexo               Sexo
edad             Edadif
departamento     Depocu
municipio        Mupocu
mes              Mesocu
dia              Diaocu
anio             Añoocu
estado_civil     Ecidif
escolaridad     Escodif
ocupacion       Ciuodif
Name: columna_en_dataset, dtype: str

Shape de modelado: X=(878136, 11), y=(878136,)


In [50]:
print('Balance de clases global:')
display(y.value_counts(normalize=True).mul(100).round(2).to_frame('%'))

print('Nulos por variable (top 10):')
display(df_model.isna().mean().sort_values(ascending=False).head(10).to_frame('%_nulos'))

Balance de clases global:


,%
grupo_causa_cie10,
Sintomas_signos_hallazgos,11.75
Endocrinas_metabolicas,10.45
Neoplasias,9.66
Isquemica_corazon,7.78
Neumonia_influenza,7.10
Accidentes_no_intencionales,6.87
Infecciosas,4.72
Homicidio,3.92
Cronica_higado,3.90


Nulos por variable (top 10):


,%_nulos
anio,0.175879
edad,0.005893
sexo,0.000000
departamento,0.000000
municipio,0.000000
mes,0.000000
dia,0.000000
estado_civil,0.000000
escolaridad,0.000000
ocupacion,0.000000


## Fase 4 - Train/Validation/Test split

Estrategia aplicada:
- 70% entrenamiento
- 15% validacion
- 15% prueba

Como la variable objetivo es categorica, se usa `stratify=y` para mantener proporcion de clases entre particiones.


In [51]:
X_train, X_temp, y_train, y_temp = train_test_split(
    df_model,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print('Shapes:')
print(f'  Train: {X_train.shape} | {y_train.shape}')
print(f'  Valid: {X_valid.shape} | {y_valid.shape}')
print(f'  Test : {X_test.shape} | {y_test.shape}')

def class_share(series: pd.Series) -> pd.Series:
    return series.value_counts(normalize=True).mul(100).round(2)

balance = pd.concat(
    [
        class_share(y_train).rename('train_%'),
        class_share(y_valid).rename('valid_%'),
        class_share(y_test).rename('test_%'),
    ],
    axis=1,
).fillna(0.0)

display(balance)

Shapes:
  Train: (614695, 11) | (614695,)
  Valid: (131720, 11) | (131720,)
  Test : (131721, 11) | (131721,)


,train_%,valid_%,test_%
grupo_causa_cie10,,,
Sintomas_signos_hallazgos,11.75,11.75,11.75
Endocrinas_metabolicas,10.45,10.45,10.45
Neoplasias,9.66,9.66,9.66
Isquemica_corazon,7.78,7.78,7.78
Neumonia_influenza,7.10,7.10,7.10
Accidentes_no_intencionales,6.87,6.87,6.87
Infecciosas,4.72,4.72,4.72
Homicidio,3.92,3.92,3.92
Cronica_higado,3.90,3.90,3.90
